# Persist daily WBGT, Tg, and Tnwb pairwise statistics

Each date is computed in a fresh child process because reusing a process caused progressive DuckDB slowdowns during long runs; limiting execution to one child at a time also bounds memory use. 

WARNING: this also allows concurrent execution across multiple dates, implemented in `scripts/daily_pairwise.py`; only do so if free system RAM is sufficient to hold concurrent child processes.

Notebook kernel performs every database write synchronously to avoid concurrent-writer contention. 

Each execution computes all three quantities and independent coverage from one wide materialized base relation. Agreement requires joint validity across WBGT, Tg, and Tnwb. Date-scoped delete-and-insert transactions make interrupted runs safe to resume and completed dates safe to rerun.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb

from liljegren_analysis import compute_daily_pairwise, persist_daily_pairwise

In [ ]:
working_directory = Path.cwd().resolve()
repository = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if repository is None:
    raise FileNotFoundError("Could not locate the repository root")

runs = (repository / ".hsb" / "runs" / "wbgt-comparison-20260823-181813",)
runs = tuple(path.resolve() for path in runs)
database_path = repository / "data" / "daily_pairwise_v5.duckdb"
start_date = date(2025, 1, 1)
end_date = date(2025, 12, 31)
threads_per_worker = 8

if start_date > end_date:
    raise ValueError("start_date must not be after end_date")
if not runs:
    raise ValueError("runs is empty")

dates = [
    start_date + timedelta(days=offset)
    for offset in range((end_date - start_date).days + 1)
]
len(dates)

## Process and persist

The executor has one live child and replaces it after every task. Daily delete-and-insert transactions make reruns resumable and idempotent.

In [ ]:
database_path.parent.mkdir(parents=True, exist_ok=True)
executor = ProcessPoolExecutor(max_workers=1, max_tasks_per_child=1)

try:
    with duckdb.connect(str(database_path)) as writer:
        for completed, result_date in enumerate(dates, start=1):
            processed_date, summaries, worker_pid, compute_seconds = (
                executor.submit(
                    compute_daily_pairwise,
                    result_date,
                    runs,
                    threads_per_worker,
                ).result()
            )
            if processed_date != result_date:
                raise RuntimeError(
                    f"Expected {result_date}, received {processed_date}"
                )

            persist_daily_pairwise(writer, result_date, summaries)
            processed_at = datetime.now(ZoneInfo("Asia/Singapore"))
            print(
                f"[{processed_at:%Y-%m-%d %H:%M:%S SGT}] "
                f"Processed {result_date} worker_pid={worker_pid} "
                f"compute={compute_seconds:.1f}s [{completed}/{len(dates)}]"
            )

        writer.execute("CHECKPOINT")
finally:
    executor.shutdown(wait=True, cancel_futures=True)

print(f"Saved daily_pairwise to {database_path}")